In [29]:
# ======= Variables =========

# Fill in the variables below and run cell first
# Note: Use forward slashes / instead of backslashes \ in file paths

DATASET_DIR = "../dataset"
OUTPUT_DIR = f"{DATASET_DIR}/original_sheets"

In [30]:
# Read the dim_patient, dim_physician and fact_txn DataFrames

from pathlib import Path

import pandas as pd

output_path = Path(OUTPUT_DIR)

dim_patient = pd.read_csv(output_path / "dim_patient.csv")
dim_physician = pd.read_csv(output_path / "dim_physician.csv")
fact_txn = pd.read_csv(output_path / "fact_txn.csv")

model_table_original = pd.read_csv(output_path / "model_table.csv")

print(f"dim_patient: rows={dim_patient.shape[0]}, columns={dim_patient.shape[1]}")
print(f"dim_physician: rows={dim_physician.shape[0]}, columns={dim_physician.shape[1]}")
print(f"fact_txn: rows={fact_txn.shape[0]}, columns={fact_txn.shape[1]}")
print(
    f"model_table_original: rows={model_table_original.shape[0]}, columns={model_table_original.shape[1]}"
)

dim_patient: rows=4020, columns=3
dim_physician: rows=25343, columns=5
fact_txn: rows=115274, columns=7
model_table_original: rows=9, columns=4


In [6]:
# Check whether there is a single patient with more than one physician in the fact_txn table
patient_physician_counts = fact_txn.groupby("PATIENT_ID")["PHYSICIAN_ID"].nunique()
patients_with_multiple_physicians = patient_physician_counts[patient_physician_counts > 1]
patients_with_multiple_physicians

PATIENT_ID
9        2
11       2
14       3
24       2
28       5
        ..
4016    26
4017    53
4018    25
4019    10
4020    10
Name: PHYSICIAN_ID, Length: 3264, dtype: int64

In [43]:
# Transform fact_txn into a patient-level transaction structure
import sys
from importlib import reload
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
import ml_antiviral_diagnosis.de as de

reload(de)

patient_transactions_df = de.transform_fact_txn_to_patient_transactions(fact_txn)

print(
    f"patient_transactions_df: rows={patient_transactions_df.shape[0]}, columns={patient_transactions_df.shape[1]}"
)
display(patient_transactions_df.head())
print("Sample transactions for the first patient:")
patient_transactions_df.iloc[0]["transactions_by_type"]

patient_transactions_df: rows=4020, columns=2


,patient_id,transactions_by_type
0,1,"{'SYMPTOMS': [], 'CONDITIONS': [{'txn_dt': '20..."
1,2,"{'SYMPTOMS': [{'txn_dt': '2022-06-22', 'physic..."
2,3,"{'SYMPTOMS': [], 'CONDITIONS': [{'txn_dt': '20..."
3,4,"{'SYMPTOMS': [], 'CONDITIONS': [{'txn_dt': '20..."
4,5,"{'SYMPTOMS': [], 'CONDITIONS': [{'txn_dt': '20..."


Sample transactions for the first patient:


{'SYMPTOMS': [],
 'CONDITIONS': [{'txn_dt': '2022-06-11',
   'physician_id': 24633,
   'txn_location_type': 'EMERGENCY ROOM - HOSPITAL',
   'insurance_type': 'COMMERCIAL',
   'txn_desc': 'disease_x'}],
 'CONTRAINDICATIONS': [],
 'TREATMENTS': []}

In [44]:
# Find the first diagnosis date for each patient and keep only transactions that occurred on or before that date
# Add TARGET column based on whether patient received Drug A or not
diagnosis_dataset_df = de.build_patient_diagnosis_dataset(patient_transactions_df)

print(
    f"diagnosis_dataset_df: rows={diagnosis_dataset_df.shape[0]}, columns={diagnosis_dataset_df.shape[1]}"
)
display(diagnosis_dataset_df.head())

diagnosis_dataset_df: rows=4020, columns=4


,patient_id,first_diagnosis_date,transactions_by_type,TARGET
0,1,2022-06-11,"{'SYMPTOMS': [], 'CONDITIONS': [{'txn_dt': '20...",0
1,2,2022-06-22,"{'SYMPTOMS': [{'txn_dt': '2022-06-22', 'physic...",0
2,3,2022-06-20,"{'SYMPTOMS': [], 'CONDITIONS': [{'txn_dt': '20...",0
3,4,2022-06-30,"{'SYMPTOMS': [], 'CONDITIONS': [{'txn_dt': '20...",0
4,5,2022-06-02,"{'SYMPTOMS': [], 'CONDITIONS': [{'txn_dt': '20...",0


In [56]:
# Save the patient diagnosis dataset to CSV
DIAGNOSIS_DATASET_FILENAME = "diagnosis_dataset.csv"
diagnosis_dataset_df.to_csv(f"{DATASET_DIR}/{DIAGNOSIS_DATASET_FILENAME}", index=False)
print(f"Patient diagnosis dataset saved to {DATASET_DIR}/{DIAGNOSIS_DATASET_FILENAME}")

Patient diagnosis dataset saved to ../dataset/diagnosis_dataset.csv


In [38]:
diagnosis_dataset_df.iloc[1339]

patient_id                                                           1340
first_diagnosis_date                                           2022-06-20
transactions_by_type    {'SYMPTOMS': [{'txn_dt': '2019-09-01', 'physic...
TARGET                                                                  0
Name: 1339, dtype: object

In [42]:
# Verify that the diagnosis dataset still contains every patient from the patient-level transaction table
missing_patient_ids = sorted(
    set(patient_transactions_df["patient_id"]) - set(diagnosis_dataset_df["patient_id"])
)
print(f"patient_transactions_df rows: {patient_transactions_df.shape[0]}")
print(f"diagnosis_dataset_df rows: {diagnosis_dataset_df.shape[0]}")
print(f"patients missing from diagnosis_dataset_df: {len(missing_patient_ids)}")
assert not missing_patient_ids, "Some patients are missing a Disease X diagnosis."

print("==========================================")

# Sanity check one patient row after the diagnosis cutoff
sample_patient = diagnosis_dataset_df.iloc[1338]
print(sample_patient[["patient_id", "first_diagnosis_date", "TARGET"]])

transaction_count = 0
for txn_type, txns in sample_patient["transactions_by_type"].items():
    print(f"Transaction type: {txn_type}, count: {len(txns)}")
    transaction_count += len(txns)

print(f"Total transactions for sample patient: {transaction_count}")
sample_patient["transactions_by_type"]

patient_transactions_df rows: 4020
diagnosis_dataset_df rows: 4020
patients missing from diagnosis_dataset_df: 0
patient_id                    1339
first_diagnosis_date    2022-06-18
TARGET                           0
Name: 1338, dtype: object
Transaction type: SYMPTOMS, count: 9
Transaction type: CONDITIONS, count: 110
Transaction type: CONTRAINDICATIONS, count: 8
Transaction type: TREATMENTS, count: 0
Total transactions for sample patient: 127


{'SYMPTOMS': [{'txn_dt': '2018-07-21',
   'physician_id': 17274,
   'txn_location_type': 'EMERGENCY ROOM - HOSPITAL',
   'insurance_type': 'COMMERCIAL',
   'txn_desc': 'sore_throat'},
  {'txn_dt': '2018-07-21',
   'physician_id': 17274,
   'txn_location_type': 'HOSPITAL OUTPATIENT',
   'insurance_type': 'COMMERCIAL',
   'txn_desc': 'SORE_THROAT'},
  {'txn_dt': '2018-07-22',
   'physician_id': 12048,
   'txn_location_type': 'EMERGENCY ROOM - HOSPITAL',
   'insurance_type': 'COMMERCIAL',
   'txn_desc': 'SORE_THROAT'},
  {'txn_dt': '2018-07-22',
   'physician_id': 21038,
   'txn_location_type': 'ON CAMPUS-OUTPATIENT HOSPITAL',
   'insurance_type': 'COMMERCIAL',
   'txn_desc': 'SORE_THROAT'},
  {'txn_dt': '2018-07-23',
   'physician_id': 13686,
   'txn_location_type': 'ON CAMPUS-OUTPATIENT HOSPITAL',
   'insurance_type': 'COMMERCIAL',
   'txn_desc': 'SORE_THROAT'},
  {'txn_dt': '2018-07-24',
   'physician_id': 8065,
   'txn_location_type': 'ON CAMPUS-OUTPATIENT HOSPITAL',
   'insurance_typ

In [45]:
# Build the model table columns defined in model_table.csv
model_table_df = de.build_model_table(
    diagnosis_dataset_df=diagnosis_dataset_df,
    dim_patient_df=dim_patient,
    dim_physician_df=dim_physician,
)

print(f"model_table_df: rows={model_table_df.shape[0]}, columns={model_table_df.shape[1]}")
display(model_table_df.head())

model_table_df: rows=4020, columns=9


,PATIENT_ID,TARGET,DISEASEX_DT,PATIENT_AGE,PATIENT_GENDER,NUM_CONDITIONS,PHYSICIAN_TYPE,PHYSICIAN_STATE,LOCATION_TYPE
0,1,0,2022-06-11,34,M,0,FAMILY MEDICINE,TX,EMERGENCY ROOM - HOSPITAL
1,2,0,2022-06-22,2,M,0,EMERGENCY MEDICINE,PA,EMERGENCY ROOM - HOSPITAL
2,3,0,2022-06-20,49,M,0,EMERGENCY MEDICINE,MS,OFFICE
3,4,0,2022-06-30,0,M,0,PEDIATRICS,PA,EMERGENCY ROOM - HOSPITAL
4,5,0,2022-06-02,34,M,0,NaN,NaN,INDEPENDENT LABORATORY


In [52]:
# Save the model table to CSV
MODEL_TABLE_FILENAME = "model_table.csv"
model_table_df.to_csv(f"{DATASET_DIR}/{MODEL_TABLE_FILENAME}", index=False)
print(f"Model table saved to {DATASET_DIR}/{MODEL_TABLE_FILENAME}")

Model table saved to ../dataset/model_table.csv


In [54]:
# Validate the generated model table against the provided template
expected_columns = model_table_original["Column Name"].tolist()
print(f"Expected columns: {expected_columns}")
print(f"Actual columns: {model_table_df.columns.tolist()}")
assert model_table_df.columns.tolist() == expected_columns, "Model table columns do not match the template."
assert model_table_df.shape[0] == diagnosis_dataset_df.shape[0], "Model table row count does not match diagnosis dataset."
print("Model table schema and row count checks passed.")

# Check for missing values in the model table
model_table_df.isna().sum().sort_values(ascending=False)

Expected columns: ['PATIENT_ID', 'TARGET', 'DISEASEX_DT', 'PATIENT_AGE', 'PATIENT_GENDER', 'NUM_CONDITIONS', 'PHYSICIAN_TYPE', 'PHYSICIAN_STATE', 'LOCATION_TYPE']
Actual columns: ['PATIENT_ID', 'TARGET', 'DISEASEX_DT', 'PATIENT_AGE', 'PATIENT_GENDER', 'NUM_CONDITIONS', 'PHYSICIAN_TYPE', 'PHYSICIAN_STATE', 'LOCATION_TYPE']
Model table schema and row count checks passed.


PHYSICIAN_STATE    605
PHYSICIAN_TYPE     605
PATIENT_ID           0
TARGET               0
DISEASEX_DT          0
PATIENT_GENDER       0
PATIENT_AGE          0
NUM_CONDITIONS       0
LOCATION_TYPE        0
dtype: int64

In [53]:
# Validate that the model table contains the same patients as the dim_patient table and each patient appears only once
model_table_patient_ids = set(model_table_df["PATIENT_ID"])
dim_patient_ids = set(dim_patient["PATIENT_ID"])
missing_patient_ids = sorted(dim_patient_ids - model_table_patient_ids)
duplicate_patient_ids = model_table_df["PATIENT_ID"][model_table_df["PATIENT_ID"].duplicated()].unique()
print(f"model_table_df rows: {model_table_df.shape[0]}")
print(f"patients missing from model_table_df: {len(missing_patient_ids)}")
print(f"duplicate patient IDs in model_table_df: {len(duplicate_patient_ids)}")
assert not missing_patient_ids, "Some patients are missing from the model table."
assert len(duplicate_patient_ids) == 0, "Some patients appear more than once in the model table."


model_table_df rows: 4020
patients missing from model_table_df: 0
duplicate patient IDs in model_table_df: 0


In [49]:
# Peruse one row of the model table
patient_id = 1339  # or replace with any PATIENT_ID value
patient_row = model_table_df.loc[model_table_df["PATIENT_ID"] == patient_id]

assert not patient_row.empty, f"PATIENT_ID {patient_id} not found."
display(patient_row)

,PATIENT_ID,TARGET,DISEASEX_DT,PATIENT_AGE,PATIENT_GENDER,NUM_CONDITIONS,PHYSICIAN_TYPE,PHYSICIAN_STATE,LOCATION_TYPE
1338,1339,0,2022-06-18,61,M,4,NURSE PRACTITIONER,MD,TELEHEALTH PROVIDED OTHER THAN IN PATIENT'S HOME


In [ ]:
# Basic overview
print(f"Shape: {model_table_df.shape}")
print("\nColumn dtypes:")
display(model_table_df.dtypes)

# Missing values
print("\nMissing values per column:")
display(model_table_df.isna().sum().sort_values(ascending=False))

# Numeric summary
print("\nNumeric summary:")
display(model_table_df.describe(include=[ "number" ]).T)

# Categorical summary
print("\nCategorical summary:")
display(model_table_df.describe(include=[ "object" ]).T)

# Target distribution
print("\nTARGET distribution:")
target_counts = model_table_df["TARGET"].value_counts(dropna=False).sort_index()
target_pct = (target_counts / len(model_table_df) * 100).round(2)
display(
    pd.DataFrame(
        {"count": target_counts, "percent": target_pct}
    )
)

# Top values for key categorical columns
for col in ["PATIENT_GENDER", "PHYSICIAN_TYPE", "PHYSICIAN_STATE", "LOCATION_TYPE"]:
    print(f"\nTop 10 values for {col}:")
    display(model_table_df[col].value_counts(dropna=False).head(10))